# Black-Scholes

This notebook mirrors the upstream `BlackScholes.ipynb` diagnostic route while using the
refactored `time_causal_vae` package. It supports two modes: released-checkpoint
reproduction and inspection of the latest local full Black-Scholes run.


## Setup and plotting style


In [ ]:
import json
import os
import shlex
from pathlib import Path

MODE = "local"  # choices: "released", "local"
SEED = 99
N_SAMPLE_TEST = 5000
RUN_SIGNATURE_DIAGNOSTIC = True
RUN_FINANCE_DIAGNOSTICS = True
RUN_AWD_DIAGNOSTIC = False
MEAN_VARIANCE_INPUT = "real"  # upstream passes real_data; use "fake" for generated paths

os.environ.setdefault("MPLCONFIGDIR", "/tmp/time-causal-vae-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from time_causal_vae.evaluation.style import apply_source_style  # noqa: E402

apply_source_style()


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root from a notebook or shell directory."""
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        has_project = (candidate / "pyproject.toml").exists()
        has_configs = (candidate / "configs" / "experiments").exists()
        if has_project and has_configs:
            return candidate
    msg = "Could not locate the time-causal-vae repository root."
    raise RuntimeError(msg)


def resolve_path(path: str | Path) -> Path:
    """Resolve a path relative to the repository root."""
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    """Render a path relative to the repository root when possible."""
    candidate = Path(path).resolve()
    try:
        return str(candidate.relative_to(REPO_ROOT))
    except ValueError:
        return str(candidate)


def latest_local_final_model(output_root: Path) -> Path | None:
    """Return the newest local BetaCVAE final_model directory."""
    candidates = sorted(
        output_root.glob("BetaCVAE_training_*/final_model"),
        key=lambda candidate: candidate.stat().st_mtime,
    )
    return candidates[-1] if candidates else None


def load_json(path: Path) -> dict:
    """Load a JSON artifact if it exists."""
    if path.exists():
        return json.loads(path.read_text())
    return {}


def squeeze_paths(tensor: torch.Tensor) -> torch.Tensor:
    """Convert path tensors to two-dimensional CPU tensors for plotting."""
    values = tensor.detach().cpu().float()
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[..., 0]
    return values


REPO_ROOT = find_repo_root()
CONFIG_PATH = resolve_path("configs/experiments/black_scholes_beta_cvae.yaml")
RELEASED_MODEL_DIR = resolve_path(
    "../TimeCausalVAE/trained_models/BSprice_timestep_60/"
    "model_BetaCVAE_De_CLSTMRes_En_CLSTMRes_Prior_RealNVP_Con_Id_Dis_None_comment_None/"
    "BetaCVAE_training_2024-08-14_14-58-49/final_model"
)
LOCAL_OUTPUT_ROOT = resolve_path("outputs/full/black_scholes_beta_cvae")
BASE_DATA_DIR = resolve_path("data/processed")

if MODE not in {"released", "local"}:
    raise ValueError("MODE must be either 'released' or 'local'.")

if N_SAMPLE_TEST != 5000 or SEED != 99:
    print(
        "Warning: upstream BlackScholes.ipynb uses n_sample_test=5000 and seed=99. "
        "Current results are useful for debugging but not directly comparable."
    )

print(f"Repository root: {REPO_ROOT}")
print(f"Mode: {MODE}")
print(f"Seed: {SEED}")
print(f"n_sample_test: {N_SAMPLE_TEST}")
if not RUN_AWD_DIAGNOSTIC:
    print("SAWD/AWD diagnostic is disabled by default; it can take many minutes on CPU.")

## Configuration and checkpoint selection


In [ ]:
if MODE == "released":
    model_dir = RELEASED_MODEL_DIR
    evaluation_dir = resolve_path("outputs/released_target_eval_bs_5000")
    experiment_dir = evaluation_dir
else:
    model_dir = latest_local_final_model(LOCAL_OUTPUT_ROOT)
    if model_dir is None:
        model_dir = LOCAL_OUTPUT_ROOT / "BetaCVAE_training_<timestamp>" / "final_model"
    evaluation_dir = LOCAL_OUTPUT_ROOT / "evaluation"
    experiment_dir = LOCAL_OUTPUT_ROOT

evaluation_dir.mkdir(parents=True, exist_ok=True)
summary_path = evaluation_dir / "summary.json"
batch_path = evaluation_dir / "evaluation_batch.pt"
hyper_metric_path = evaluation_dir / "hyper_metric.pkl"
selected_model_path = experiment_dir / "selected_model.json"

print(f"config: {display_path(CONFIG_PATH)}")
print(f"model_dir: {display_path(model_dir)}")
print(f"evaluation_dir: {display_path(evaluation_dir)}")
print(f"model_dir exists: {model_dir.exists()}")

selected_config = yaml.safe_load(CONFIG_PATH.read_text())
rows = [
    ("dataset", selected_config.get("dataset", {}).get("name")),
    ("objective", selected_config.get("model", {}).get("objective")),
    ("encoder", selected_config.get("model", {}).get("encoder")),
    ("decoder", selected_config.get("model", {}).get("decoder")),
    ("conditioner", selected_config.get("model", {}).get("conditioner")),
    ("prior", selected_config.get("model", {}).get("prior")),
    ("beta", selected_config.get("model", {}).get("beta")),
    ("epochs", selected_config.get("training", {}).get("epochs")),
    ("n_sample", selected_config.get("dataset", {}).get("n_sample")),
    ("n_timestep", selected_config.get("dataset", {}).get("n_timestep")),
]
display(pd.DataFrame(rows, columns=["field", "value"]))

command = [
    "poetry",
    "run",
    "tcvae-evaluate",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    display_path(model_dir),
    "--output-dir",
    display_path(evaluation_dir),
    "--n-sample-test",
    str(N_SAMPLE_TEST),
    "--seed",
    str(SEED),
]
print("Equivalent evaluation command:")
print(" ".join(shlex.quote(part) for part in command))

## Load model and generate real/fake/reconstruction data


In [ ]:
from time_causal_vae.evaluation.checkpoints import TargetModelEvaluator

if not model_dir.exists():
    evaluator = None
    real_data = fake_data = recon_data = None
    print("Checkpoint not found. Switch MODE or train/evaluate a local checkpoint first.")
else:
    evaluator = TargetModelEvaluator(str(model_dir), base_data_dir=str(BASE_DATA_DIR))
    real_data, fake_data, recon_data = evaluator.load_data(
        n_sample_test=N_SAMPLE_TEST,
        seed=SEED,
    )
    base_dataset = evaluator.ensure_base_dataset()
    horizon = base_dataset.dt * base_dataset.n_timestep
    batch = {
        "real_data": real_data.detach().cpu(),
        "fake_data": fake_data.detach().cpu(),
        "recon_data": recon_data.detach().cpu(),
    }
    display(
        pd.DataFrame(
            [
                {
                    "tensor": name,
                    "shape": tuple(tensor.shape),
                    "mean": float(tensor.float().mean()),
                    "std": float(tensor.float().std()),
                    "min": float(tensor.float().min()),
                    "max": float(tensor.float().max()),
                }
                for name, tensor in batch.items()
            ]
        )
    )
    print(f"Black-Scholes dt: {base_dataset.dt}")
    print(f"Black-Scholes n_timestep: {base_dataset.n_timestep}")
    print(f"diagnostic horizon: {horizon}")

## Path comparison


In [ ]:
if real_data is None:
    print("Path comparison skipped because no checkpoint was loaded.")
else:
    apply_source_style()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharex=True, sharey=True)
    plot_specs = [
        ("real_data", real_data, "Real"),
        ("fake_data", fake_data, "Generated"),
        ("recon_data", recon_data, "Reconstruction"),
    ]
    for axis, (_key, values, title) in zip(axes, plot_specs, strict=True):
        paths = squeeze_paths(values)[:24]
        axis.plot(paths.T, alpha=0.45, linewidth=0.9)
        axis.set_title(title)
        axis.set_xlabel("Time")
    axes[0].set_ylabel("Price")
    fig.tight_layout()

    real = squeeze_paths(real_data)
    fake = squeeze_paths(fake_data)
    recon = squeeze_paths(recon_data)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for label, values in {"real": real, "fake": fake, "recon": recon}.items():
        axes[0].hist(values[:, -1].numpy(), bins=60, alpha=0.35, density=True, label=label)
        log_returns = torch.log(values[:, 1:] / values[:, :-1]).flatten()
        axes[1].hist(log_returns.numpy(), bins=80, alpha=0.35, density=True, label=label)
    axes[0].set_title("Terminal price")
    axes[1].set_title("Log returns")
    for axis in axes:
        axis.legend()
    fig.tight_layout()

## Drift and volatility


In [ ]:
if real_data is None:
    print("Drift-volatility diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.external.drift_volatility import (
        compare_drift_volatility,
    )
    from time_causal_vae.evaluation.volatility import volatility_diagnostics

    drift_volatility_path = evaluation_dir / "drift_volatility.png"
    compare_drift_volatility(real_data, fake_data, T=horizon, file_path=drift_volatility_path)
    volatility_summary = volatility_diagnostics(real_data, fake_data, horizon)
    display(pd.DataFrame([volatility_summary]).T.rename(columns={0: "value"}))

## Weak metrics: MMD and SWD


In [ ]:
if real_data is None:
    print("Weak metric diagnostic skipped because no checkpoint was loaded.")
else:
    hyper_metric = evaluator.compute_hyper_metric(real_data, fake_data)
    metric_summary = {key: float(value.detach().cpu()) for key, value in hyper_metric.items()}
    display(pd.DataFrame([metric_summary]))

    if RUN_SIGNATURE_DIAGNOSTIC:
        from time_causal_vae.evaluation.metrics import SignatureMMD

        try:
            signature_mmd = SignatureMMD()(real_data, fake_data)
            print(f"Signature MMD: {float(signature_mmd.detach().cpu())}")
        except ModuleNotFoundError as exc:
            print(f"Expected-signature diagnostic skipped: {exc}")
    else:
        print(
            "Expected-signature diagnostic skipped. signatory is not part of the "
            "standard environment for the current Python/PyTorch constraints."
        )

## Log-utility


In [ ]:
if not RUN_FINANCE_DIAGNOSTICS:
    print("Log-utility diagnostic skipped. Set RUN_FINANCE_DIAGNOSTICS=True to run it.")
elif evaluator is None:
    print("Log-utility diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.finance.log_utility import compare_log_utility_max

    compare_log_utility_max(evaluator, file_path=evaluation_dir / "log_utility.png")

## Mean-variance portfolio


In [ ]:
if not RUN_FINANCE_DIAGNOSTICS:
    print("Mean-variance diagnostic skipped. Set RUN_FINANCE_DIAGNOSTICS=True to run it.")
elif evaluator is None:
    print("Mean-variance diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.finance.mean_variance import (
        compute_mean_variance,
        plot_mean_variance,
    )

    if MEAN_VARIANCE_INPUT not in {"real", "fake"}:
        raise ValueError("MEAN_VARIANCE_INPUT must be either real or fake.")
    mv_data = real_data if MEAN_VARIANCE_INPUT == "real" else fake_data
    print(
        "Mean-variance input:",
        MEAN_VARIANCE_INPUT,
        "(upstream BlackScholes.ipynb uses real_data even though the curve is labelled fake)",
    )
    sigma_strategy, sigma_strategy_data = compute_mean_variance(evaluator, mv_data)
    plot_mean_variance(
        sigma_strategy,
        sigma_strategy_data,
        base_dataset.sigma,
        file_path=evaluation_dir / "mean_variance_portfolio.png",
    )

## Optimal stopping


In [ ]:
if not RUN_FINANCE_DIAGNOSTICS:
    print("Optimal-stopping diagnostic skipped. Set RUN_FINANCE_DIAGNOSTICS=True to run it.")
elif evaluator is None:
    print("Optimal-stopping diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.finance.optimal_stopping import (
        compare_os,
        load_os_data,
    )

    real_os, fake_os, control_os = load_os_data(evaluator, 100, 12, 5000)
    compare_os(
        r=base_dataset.mu,
        s0=100,
        n_maturity_timestep=12,
        dt=base_dataset.dt,
        strike=100,
        real_data_list=real_os,
        fake_data_list=fake_os,
        control_data_dict=control_os,
        file_path=evaluation_dir / "optimal_stopping.png",
    )

## Optional adapted Wasserstein / SAWD


In [ ]:
if not RUN_AWD_DIAGNOSTIC:
    print("SAWD/AWD diagnostic skipped. It can take many minutes on CPU.")
elif evaluator is None:
    print("SAWD/AWD diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.external.unconditional import (
        compute_eval_awd_dist_uncon,
        load_data_eval_dist_uncon,
        plot_eval_awd_dist_uncon,
    )

    fake_data_list, control_data_dict = load_data_eval_dist_uncon(evaluator)
    control_data = next(iter(control_data_dict.values()))
    sawd_dist = compute_eval_awd_dist_uncon(
        real_data,
        fake_data,
        control_data,
        evaluation_dir,
    )
    plot_eval_awd_dist_uncon(sawd_dist, evaluation_dir)